<img src="https://github.com/IOAI-official/IOAI-2025/blob/main/Individual-Contest/Radar/figs/IOAI-Logo.png?raw=1" alt="IOAI Logo" width="200" height="auto">

[IOAI 2025 (Beijing, China), Individual Contest](https://ioai-official.org/china-2025)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IOAI-official/IOAI-2025/blob/main/Individual-Contest/Radar/Radar.ipynb)

# Radar

## 1. Problem Description

Radar is a key technology in wireless communication, with widespread applications such as self-driving cars. It typically involves an antenna that transmits specific signals and receives their reflections from objects in the environment. By processing these signals, the system determines the angular direction, distance, and velocity of target objects.

In real-world applications, radar signal processing is challenging due to noise and reflections from non-target objects in the surroundings. For example, when attempting to detect pedestrians, the radar may also receive reflections from trees or other background objects, which can degrade accuracy. Your task is to use AI to analyze the signals received by the radar and identify the presence of a human at each position.

In this task, we provide an **indoor radar experiment dataset**, and your objective is to develop a model that performs **radar semantic segmentation**.


## 2. Dataset

To measure objects surrounding a radar, the following key parameters are used:

- **Range**: The straight-line distance between the radar and an object.
- **Azimuth**: The horizontal angle (left to right) between the radar and the object.
- **Elevation**: The vertical angle (up or down) of the object relative to the radar.
- **Velocity**: The speed at which the object is moving toward or away from the radar.

<img src="https://github.com/IOAI-official/IOAI-2025/blob/main/Individual-Contest/Radar/figs/Radar%20Fig%201.png?raw=1" width="300">


The radar data is processed into multiple **heatmaps**, each encoding the **received signal strength** at various positions and directions.

- **Static heatmaps** emphasize reflections from **stationary** objects.
- **Dynamic heatmaps** highlight changes caused by **moving** objects.

When no object is present at a specific location, the signal consists mostly of background noise and appears weak. In contrast, reflections from an object increase signal intensity, enabling detection of the object.

For example, the **static range-azimuth heatmap** represents signal strength across different distances (**range**) and horizontal angles (**azimuth**), mainly reflected by stationary objects.

Each sample in the dataset is stored in a `.mat.pt` file as a tensor of shape $7 \times 50 \times 181$, where:

- 7 is the number of maps (6 heatmaps + 1 semantic label map),
- 50 represents range bins (distance),
- 181 represents angular or velocity bins, covering angles from \-90° to \+90° in either the horizontal or vertical plane. You can assume that the velocity bins are also remapped from \-90° to \+90° for visualization consistency.
- each heatmap intensity value is normalized to [0, 1], representing received signal strength.

The 6 heatmaps are structured as follows:

- **Index 0**: Static range-azimuth heatmap
- **Index 1**: Dynamic range-azimuth heatmap
- **Index 2**: Static range-elevation heatmap
- **Index 3**: Dynamic range-elevation heatmap
- **Index 4**: Static range-velocity heatmap
- **Index 5**: Dynamic range-velocity heatmap

All values in heatmaps are **normalized**, so no unit conversion is required.

The **map at Index 6** is the semantic label map, stored in range-azimuth format.

- **-1**: Background (no target)
- **0**: Suitcase
- **1**: Chair
- **2**: Human
- **3**: Wall

This is the visualization of 1.mat.pt in training_set:

<img src="https://github.com/IOAI-official/IOAI-2025/blob/main/Individual-Contest/Radar/figs/Radar%20Fig%202.png?raw=1" width="675">

Here is part of a sample from the dataset:

<img src="https://github.com/IOAI-official/IOAI-2025/blob/main/Individual-Contest/Radar/figs/Radar%20Fig%203.png?raw=1" width="675">


Data scale: 1800 samples in the training set, 500 samples in the validation set, and 500 samples in the test set.

## 3\. Task

Your task is to develop a model that takes the **first six heatmaps** (indices 0 to 5) as input, and predicts the **semantic label map** (index 6) as the output. The goal is to accurately identify what the target is(-1 to 3) at each location in the radar’s field of view.

1. **Input**: A tensor of shape $6 \times 50 \times 181$, representing six radar heatmaps.
2. **Output**: A tensor of shape $50 \times 181$, representing the target semantic label map.


## 4\. Submission

Please submit a file named `submission.ipynb`. The output is a zip file named "submission.zip", which contains two tables `submission_val.csv` and `submission_test.csv` corresponding to the prediction results of the validation set and the test set respectively.

**Note:** The output table should have a header, the data in the table is not the actual solved data, it is only used as an example of the submission format.

| filename | pixel_0 | pixel_1 | ... | pixel_9049 |
| :------: | :-----: | ------- | --- | ---------- |
| 1.mat.pt |   -1    | -1      | ... | -1         |
|   ...    |   ...   | ...     | ... | ...        |

## 5\. Score

The score is based on the **accuracy of label recognition**. Correctly identifying target points is weighted more heavily than correctly identifying background points.

### Scoring Criteria:

* Each correctly identified **background pixel** earns **1 point**.

* Each correctly identified **non-background pixel** earns **50 points**.

* The final score is normalized to a **0-1 point** by comparing it to the maximum possible score.

### Formula：
$$
Score = \frac{|C_{0,correct}| \times 1 + |C_{1,correct}| \times bonus}{|C_0| \times 1 + |C_1| \times bonus}
$$
where:

$$
\begin{aligned}
I &= \{1, 2, \dots, 50\times 181\}\\
C_0 &= \{i \in I \mid y_i = -1\}\\
C_1 &= \{i \in I \mid y_i \neq -1\}\\
C_{0,correct} &= \{i \in C_0 \mid p_i = y_i\}\\
C_{1,correct} &= \{i \in C_1 \mid p_i = y_i\}\\
\end{aligned}
$$


### Example

For a $3\times3$ heatmap, assume the Ground Truth is:

$$
\begin{bmatrix}
-1 & -1 & -1 \\
1 & 2 & 3 \\
-1 & -1 & -1
\end{bmatrix}
$$

The intenteded result is:

$$
\begin{bmatrix}
-1 & 1 & -1 \\
-1 & 2 & -1 \\
-1 & 3 & -1
\end{bmatrix}
$$

Then there are four correctly identified `-1` and one correctly identified `2`. Your score is 4 + 50 = 54 points. The maximum possible score is 6 + 50 * 3 = 156, that is, the score for six background pixels and three non-background pixels. Your normalized score is 54 / 156 = 0.346.

$$
Score = \frac{4 \times 1 + 1 \times 50}{6 \times 1 + 3 \times 50}=0.346
$$

## 6. Baseline and Training Set

- Below you can find the baseline solution.
- The dataset is in `training_set` folder.
- The highest score by the Scientific Committee for this task is 0.90 in Leaderboard B, this score is used for score unification.
- The baseline score by the Scientific Committee for this task is 0.67 in Leaderboard B, this score is used for score unification.

### Data Loading

In [22]:
!git clone https://github.com/IOAI-official/IOAI-2025.git
# Move the training_set folder from the cloned repository to the current directory
!mv IOAI-2025/Individual-Contest/Radar/training_set ./training_set
# Move the figs folder from the cloned repository to the current directory
!mv IOAI-2025/Individual-Contest/Radar/figs ./figs
# Move the validation_set from the solution folder to the local Solution directory
!mv IOAI-2025/Individual-Contest/Radar/Solution/validation_set ./Solution/validation_set
# Move the test_set from the solution folder to the local Solution directory
!mv IOAI-2025/Individual-Contest/Radar/Solution/test_set ./Solution/test_set
# Create a directory for Scoring within the Solution folder if it doesn't exist
!mkdir -p Solution/Scoring
# Move all files from the Scoring folder in the cloned repository to the local Scoring directory
!mv IOAI-2025/Individual-Contest/Radar/Solution/Scoring/* ./Solution/Scoring/

fatal: destination path 'IOAI-2025' already exists and is not an empty directory.
mv: cannot stat 'IOAI-2025/Individual-Contest/Radar/training_set': No such file or directory
mv: cannot stat 'IOAI-2025/Individual-Contest/Radar/figs': No such file or directory
mv: cannot stat 'IOAI-2025/Individual-Contest/Radar/Solution/validation_set': No such file or directory
mv: cannot stat 'IOAI-2025/Individual-Contest/Radar/Solution/test_set': No such file or directory
mv: cannot stat 'IOAI-2025/Individual-Contest/Radar/Solution/Scoring/*': No such file or directory


In [23]:
import random
import numpy as np
import torch

# Set a random seed for reproducibility across different runs
# This ensures that random operations (like splitting data or initializing weights)
# produce the same results each time the code is executed, which is crucial for debugging and comparison.
seed = 42

random.seed(seed)                  # Set seed for Python's built-in random module
np.random.seed(seed)               # Set seed for NumPy, a library for numerical operations
torch.manual_seed(seed)            # Set seed for PyTorch on the CPU
torch.cuda.manual_seed(seed)       # Set seed for PyTorch on a single GPU
torch.cuda.manual_seed_all(seed)   # Set seed for PyTorch on all GPUs (if multiple are available)

# Ensures deterministic behavior for cuDNN, a GPU-accelerated deep learning library.
# This can sometimes slightly reduce performance but guarantees consistent results.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [24]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# CustomDataset class for loading training and validation data
# It inherits from torch.utils.data.Dataset, which is a standard PyTorch class for datasets.
class CustomDataset(Dataset):
    def __init__(self, file_paths, transform=None):
        # Initialize the dataset with a list of file paths and an optional transform function
        self.file_paths = file_paths
        self.transform = transform
        # Extract just the filenames from the full paths for later use (e.g., in CSV output)
        self.file_names = [os.path.basename(path) for path in file_paths]

    def __len__(self):
        # Return the total number of samples in the dataset
        return len(self.file_paths)

    def __getitem__(self, idx):
        # This method is called to load a single sample at a given index (idx)
        # Load the data from the .mat.pt file using torch.load
        data = torch.load(self.file_paths[idx], weights_only=True)

        # The first 6 channels are image data (heatmaps)
        images = data[:6]
        # The 7th channel (index 6) is the semantic label map
        labels = data[6]

        # Convert image data to float type (common for neural network inputs)
        images = images.float()
        # Convert labels to long type (common for classification targets in PyTorch)
        labels = labels.long()
        # Adjust labels: original labels are -1 to 3. Add 1 to make them 0 to 4.
        # This is often done because CrossEntropyLoss expects non-negative class indices.
        labels = labels + 1

        # Apply any specified transformations to images and labels
        if self.transform:
            images = self.transform(images)
            labels = self.transform(labels)

        # Return the processed images, labels, and the original filename
        return images, labels, self.file_names[idx]

# CustomDataset_test class for loading test data (without labels)
# This is similar to CustomDataset but only loads images, as test data doesn't have ground truth labels.
class CustomDataset_test(Dataset):
    def __init__(self, file_paths, transform=None):
        self.file_paths = file_paths
        self.transform = transform
        self.file_names = [os.path.basename(path) for path in file_paths]

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        data = torch.load(self.file_paths[idx], weights_only=True)

        images = data[:6] # Only load the image data (first 6 channels)

        images = images.float()

        if self.transform:
            images = self.transform(images)

        # For test data, we only return images and filenames, no labels
        return images, self.file_names[idx]

# Function to generate a list of file paths from a given base directory
def generate_file_paths(base_path):
    file_paths = []
    # Iterate through each item in the directory
    for frame in os.listdir(base_path):
        # Construct the full path to the file
        frame_path = os.path.join(base_path, frame)
        # Check if the file is a .mat.pt file (our data format)
        if frame_path.endswith('.mat.pt'):
            file_paths.append(frame_path)
    # Filter out any paths that might not actually exist (though os.listdir usually ensures this)
    return [path for path in file_paths if os.path.exists(path)]

# Function to load data and create DataLoader instances for training and testing
def load_data(base_path, batch_size=4, num_workers=2, test_size=0.2):
    # Get all .mat.pt file paths from the base_path
    file_paths = generate_file_paths(base_path)

    # Split the file paths into training and testing sets
    # test_size determines the proportion of the dataset to include in the test split (e.g., 0.2 means 20% for testing)
    # random_state ensures the split is reproducible
    train_paths, test_paths = train_test_split(file_paths, test_size=test_size, random_state=42)

    # Create dataset objects for training and testing using our CustomDataset class
    train_dataset = CustomDataset(file_paths=train_paths)
    test_dataset = CustomDataset(file_paths=test_paths)

    # Create DataLoader objects for both datasets
    # DataLoader helps in iterating over the dataset in batches, shuffling data, and parallel data loading.
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size, # Number of samples per batch
        shuffle=True,          # Shuffle data each epoch for training
        num_workers=num_workers, # Number of subprocesses to use for data loading
        drop_last=True         # Drop the last incomplete batch if dataset size is not divisible by batch_size
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,         # No need to shuffle test data
        num_workers=num_workers,
        drop_last=True
    )

    return train_loader, test_loader

### Model Definition and Training

In [29]:
import torch
import torch.nn as nn
import torch.optim as optim

# Define our neural network model. This is a simple Convolutional Neural Network (CNN).
# It inherits from nn.Module, the base class for all neural network modules in PyTorch.
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__() # Call the constructor of the parent class
        # First convolutional layer: 6 input channels (our heatmaps) to 16 output channels.
        # kernel_size=3 means a 3x3 filter. padding=1 helps maintain the spatial dimensions of the input.
        self.conv1 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=3, padding=1)
        # Second convolutional layer: 16 input channels to 32 output channels.
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        # Third convolutional layer: 32 input channels to 5 output channels.
        # We have 5 classes (background -1, suitcase 0, chair 1, human 2, wall 3). After adding 1, they become 0-4.
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=5, kernel_size=3, padding=1)

        # Rectified Linear Unit (ReLU) activation function. It introduces non-linearity.
        self.relu = nn.ReLU()

class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=5, kernel_size=3, padding=1)
        self.relu = nn.ReLU()

    def forward(self, x):
      x = self.relu(self.bn1(self.conv1(x)))
      x = self.relu(self.bn2(self.conv2(x)))
      x = self.conv3(x)
      return x

# Function to train the model
def train(model, train_loader, test_loader, optimizer, criterion, num_epochs=100):
    train_losses = [] # List to store training losses per epoch
    val_losses = []   # List to store validation losses per epoch

    for epoch in range(num_epochs):
        model.train() # Set the model to training mode (enables dropout, batch norm etc.)
        epoch_loss = 0.0
        batch_count = 0

        # Iterate over batches in the training data loader
        for images, labels, _ in train_loader:
            # Move data to GPU if available, otherwise keep on CPU
            images = images.cuda() if torch.cuda.is_available() else images
            labels = labels.cuda() if torch.cuda.is_available() else labels

            optimizer.zero_grad() # Clear previous gradients

            outputs = model(images) # Forward pass: get model predictions
            # Reshape outputs for CrossEntropyLoss: [Batch, Classes, Height * Width]
            outputs = outputs.view(outputs.size(0), outputs.size(1), -1)
            # Reshape labels for CrossEntropyLoss: [Batch, Height * Width]
            labels = labels.view(labels.size(0), -1)

            loss = criterion(outputs, labels) # Calculate the loss

            loss.backward() # Backward pass: compute gradients
            optimizer.step() # Update model parameters

            epoch_loss += loss.item() # Accumulate loss for the current epoch
            batch_count += 1

        avg_train_loss = epoch_loss / batch_count # Calculate average training loss for the epoch
        train_losses.append(avg_train_loss)

        model.eval() # Set the model to evaluation mode (disables dropout, batch norm etc.)
        val_loss = 0.0
        val_batch_count = 0

        # Disable gradient calculation for validation (saves memory and speeds up computation)
        with torch.no_grad():
            # Iterate over batches in the validation data loader
            for images, labels, _ in test_loader:
                images = images.cuda() if torch.cuda.is_available() else images
                labels = labels.cuda() if torch.cuda.is_available() else labels

                outputs = model(images) # Forward pass
                outputs = outputs.view(outputs.size(0), outputs.size(1), -1)
                labels = labels.view(labels.size(0), -1)
                loss = criterion(outputs, labels) # Calculate loss

                val_loss += loss.item() # Accumulate validation loss
                val_batch_count += 1

        avg_val_loss = val_loss / val_batch_count # Calculate average validation loss
        val_losses.append(avg_val_loss)

        # Print training and validation loss every 2 epochs
        if (epoch+1) % 2 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], '
                  f'Train Loss: {avg_train_loss:.4f}, '
                  f'Val Loss: {avg_val_loss:.4f}')

    return train_losses, val_losses

# Define the path to the training data. This path is relative to the notebook.
TRAIN_PATH = "./"
# The training set is deployed automatically in the testing machine.
# You notebook can access the TRAIN_PATH even if you do not mount it along with notebook.
data_path = TRAIN_PATH + 'training_set'

# Load the training and validation data using the load_data function
train_loader, test_loader = load_data(
    base_path=data_path, # Path to the training dataset
    batch_size=4,        # Number of samples in each batch
    num_workers=2,       # Number of subprocesses for data loading
    test_size=0.2        # 20% of the training data will be used for validation
)

# Instantiate our model
model = MyModel()
# Move the model to GPU if a CUDA-enabled GPU is available
if torch.cuda.is_available():
    model = model.cuda()

# Define the loss function: Cross Entropy Loss is commonly used for multi-class classification
weights = torch.tensor([1.0, 50.0, 50.0, 50.0, 50.0])
if torch.cuda.is_available():
    weights = weights.cuda()
criterion = nn.CrossEntropyLoss(weight=weights)
# Define the optimizer: Adam is an adaptive learning rate optimization algorithm
optimizer = optim.Adam(model.parameters(), lr=0.001) # Optimize model parameters with a learning rate of 0.001

# Start the training process
train_losses, val_losses = train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    criterion=criterion,
    num_epochs=20 # Train for 40 epochs
)

Epoch [2/20], Train Loss: 0.4174, Val Loss: 0.4138
Epoch [4/20], Train Loss: 0.3532, Val Loss: 0.3694
Epoch [6/20], Train Loss: 0.3253, Val Loss: 0.3428
Epoch [8/20], Train Loss: 0.3107, Val Loss: 0.3482
Epoch [10/20], Train Loss: 0.3006, Val Loss: 0.3378
Epoch [12/20], Train Loss: 0.2947, Val Loss: 0.3195
Epoch [14/20], Train Loss: 0.2886, Val Loss: 0.3257
Epoch [16/20], Train Loss: 0.2807, Val Loss: 0.3222
Epoch [18/20], Train Loss: 0.2785, Val Loss: 0.3120
Epoch [20/20], Train Loss: 0.2752, Val Loss: 0.3202


In [30]:
# Calculate competition score
model.eval()
total_bg = 0
total_obj = 0
correct_bg = 0
correct_obj = 0

with torch.no_grad():
    for images, labels, _ in train_loader:
        if torch.cuda.is_available():
            images = images.cuda()
            labels = labels.cuda()
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        bg_mask = (labels == 0)       # background (was -1, shifted to 0)
        obj_mask = (labels != 0)      # objects

        correct_bg += (preds[bg_mask] == labels[bg_mask]).sum().item()
        correct_obj += (preds[obj_mask] == labels[obj_mask]).sum().item()
        total_bg += bg_mask.sum().item()
        total_obj += obj_mask.sum().item()

score = (correct_bg * 1 + correct_obj * 50) / (total_bg * 1 + total_obj * 50)
print(f"Score: {score:.4f}")
print(f"Background accuracy: {correct_bg/total_bg:.4f}")
print(f"Object accuracy: {correct_obj/total_obj:.4f}")

Score: 0.8958
Background accuracy: 0.9224
Object accuracy: 0.8736


### Generate CSV for Submission

In [ ]:
# Run inference on validation set and testing set
from torch.utils.data import DataLoader
import pandas as pd

# Function to run inference (make predictions) on a given dataset
def run_inference(model, data_loader):
    """Run inference and return predictions with filenames"""
    model.eval() # Set the model to evaluation mode (important for consistent results)
    predictions = [] # List to store the model's predictions
    filenames = []   # List to store the corresponding filenames

    with torch.no_grad(): # Disable gradient calculation during inference to save memory and speed up
        for images, file_names in data_loader:
            # Move images to GPU if available
            images = images.cuda() if torch.cuda.is_available() else images

            outputs = model(images) # Get model output (raw scores for each class)
            # Get the class with the highest score as the prediction
            preds = torch.argmax(outputs, dim=1)

            # Convert predictions back to original label range [-1, 3]
            # Our model output classes are 0-4 (because we added 1 to original labels -1 to 3)
            # So, subtract 1 to get them back to -1 to 3 range.
            preds = preds - 1

            # Flatten predictions for each sample in the batch and store them
            for i, pred in enumerate(preds):
                predictions.append(pred.cpu().numpy().flatten()) # Move to CPU and convert to numpy array
                filenames.append(file_names[i]) # Store the filename

    return predictions, filenames

# DATA_PATH is an environment variable that points to the validation and test sets
# In a local environment, it defaults to 'Solution/'
if os.environ.get('DATA_PATH'):
    DATA_PATH = os.environ.get("DATA_PATH") + "/"
else:
    DATA_PATH = "Solution/"  # Fallback for local testing

# Load validation set file paths and create a DataLoader
val_paths = generate_file_paths(DATA_PATH + 'validation_set')
val_dataset = CustomDataset_test(file_paths=val_paths) # Use CustomDataset_test as no labels are needed
val_loader = DataLoader(
    val_dataset,
    batch_size=1, # Process one sample at a time for inference
    shuffle=False, # No need to shuffle during inference
    num_workers=2
)

# Load testing set file paths and create a DataLoader
test_paths = generate_file_paths(DATA_PATH + 'test_set')
test_dataset = CustomDataset_test(file_paths=test_paths)
test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2
)

# Run inference on the validation set
print("Running inference on validation set...")
val_predictions, val_filenames = run_inference(model, val_loader)

# Save validation results to a CSV file
val_results = []
for filename, pred in zip(val_filenames, val_predictions):
    row = {'filename': filename} # Start a dictionary for each row with the filename
    for i, p in enumerate(pred):
        row[f'pixel_{i}'] = p # Add each predicted pixel value as a new column
    val_results.append(row)

val_df = pd.DataFrame(val_results) # Convert the list of dictionaries to a Pandas DataFrame
val_df.to_csv('submission_val.csv', index=False) # Save DataFrame to CSV without writing the DataFrame index
print(f"Validation results saved to output_validation.csv with shape: {val_df.shape}")

# Run inference on the testing set
print("Running inference on testing set...")
test_predictions, test_filenames = run_inference(model, test_loader)

# Save testing results to a CSV file
test_results = []
for filename, pred in zip(test_filenames, test_predictions):
    row = {'filename': filename}
    for i, p in enumerate(pred):
        row[f'pixel_{i}'] = p
    test_results.append(row)

test_df = pd.DataFrame(test_results)
test_df.to_csv('submission_test.csv', index=False)
print(f"Testing results saved to output_testing.csv with shape: {test_df.shape}")

print("\nInference completed! Results saved to:")
print("- submission_val.csv (for validation set leaderboard)")
print("- submission_test.csv (for testing set leaderboard)")

### Create .zip File

In [ ]:
import zipfile
import os

# Define the list of files to be included in the zip archive
files_to_zip = ['submission_val.csv', 'submission_test.csv']
# Define the name of the output zip file
zip_filename = 'submission.zip'

# Create a zip file
# 'w' mode opens the zip file for writing (creates a new one or overwrites existing)
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    # Iterate through each file in the list
    for file in files_to_zip:
        # Add each file to the zip archive
        # os.path.basename(file) ensures that only the filename (e.g., 'submission_val.csv')
        # is used inside the zip, not the full path.
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} Created successfully!')

In [ ]:
import matplotlib.pyplot as plt

model.eval() # Set the model to evaluation mode for consistent inference
with torch.no_grad(): # Disable gradient calculations to save memory and speed up computation
    # Iterate through one batch of the training data loader
    for images, labels, fname in train_loader:
        if torch.cuda.is_available():
            images = images.cuda() # Move images to GPU if available
        outputs = model(images) # Get model predictions (raw scores)
        # Get the predicted class by finding the argmax (index of highest score)
        # Subtract 1 to convert from 0-4 range back to original -1 to 3 range.
        preds = torch.argmax(outputs, dim=1) - 1
        # Also convert the ground truth labels back to -1 to 3 range for comparison
        labels = labels - 1

        # Create a figure with 1 row, 3 columns for displaying images
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        # Display the first heatmap (Index 0: Static range-azimuth) from the input images
        axes[0].imshow(images[0][0].cpu(), cmap='hot') # .cpu() moves tensor to CPU for matplotlib
        axes[0].set_title('Input') # Set title for the subplot

        # Display the Ground Truth semantic label map
        axes[1].imshow(labels[0].cpu(), cmap='tab10', vmin=-1, vmax=3) # Use 'tab10' colormap for discrete labels
        axes[1].set_title('Ground Truth')

        # Display the model's Predicted semantic label map
        axes[2].imshow(preds[0].cpu(), cmap='tab10', vmin=-1, vmax=3)
        axes[2].set_title('Prediction')

        plt.tight_layout() # Adjust subplot parameters for a tight layout
        plt.show() # Display the plot
        break # Only visualize the first batch and then exit the loop